# LightGBM

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

### Fitting the X_tr_encoded to the classifier as is before fine tuning

In [3]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import joblib

from src.config import SEED, MODEL_DIR
from src.data import load_train, get_split
from src.preprocessing import learn_preprocessing, apply_preprocessing
from src.models import make_lightgbm
from src.evaluate import report

In [4]:
X, y = load_train()
X_tr, X_val, y_tr, y_val = get_split(X, y)
print(X_tr.shape, X_val.shape)

(559708, 21) (139927, 21)


### Preprocessing - one with Numeric and other with one hot encoding

In [4]:
state_ohe = learn_preprocessing(X_tr, ratings="onehot")

X_tr_encoded = apply_preprocessing(X_tr, state_ohe)
X_val_encoded = apply_preprocessing(X_val, state_ohe)

In [5]:
state_num = learn_preprocessing(X_tr, ratings="numeric")

X_tr_num = apply_preprocessing(X_tr, state_num)
X_val_num = apply_preprocessing(X_val, state_num)

### Without scaling

In [5]:
state_ohe = learn_preprocessing(X_tr, ratings="onehot", scaling="no")

X_tr_encoded = apply_preprocessing(X_tr, state_ohe)
X_val_encoded = apply_preprocessing(X_val, state_ohe)

In [6]:
state_num = learn_preprocessing(X_tr, ratings="numeric", scaling="no")

X_tr_num = apply_preprocessing(X_tr, state_num)
X_val_num = apply_preprocessing(X_val, state_num)

### Training both One hot encoded and Numeric Encoded

In [6]:
lgbm_enc = make_lightgbm()
lgbm_enc.fit(X_tr_encoded, y_tr)

,learning_rate,0.05
,n_estimators,500
,random_state,42
,n_jobs,-1
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None


In [7]:
lgbm_num = make_lightgbm()
lgbm_num.fit(X_tr_num, y_tr)

,learning_rate,0.05
,n_estimators,500
,random_state,42
,n_jobs,-1
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None


#### The same without scaling

In [7]:
lgbm_enc_no_scale = make_lightgbm()
lgbm_enc_no_scale.fit(X_tr_encoded, y_tr)

,learning_rate,0.05
,n_estimators,500
,random_state,42
,n_jobs,-1
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None


In [8]:
lgbm_num_no_scale = make_lightgbm()
lgbm_num_no_scale.fit(X_tr_num, y_tr)

,learning_rate,0.05
,n_estimators,500
,random_state,42
,n_jobs,-1
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None


### Evaluation and Prediction

In [8]:
train_auc = roc_auc_score(
    y_tr, lgbm_enc.predict_proba(X_tr_encoded)[:, 1]
)
print(f"Train AUC: {train_auc:.4f}")

Train AUC: 0.9629


In [9]:
train_auc_2 = roc_auc_score(
    y_tr, lgbm_num.predict_proba(X_tr_num)[:, 1]
)
print(f"Train AUC: {train_auc:.4f}")

Train AUC: 0.9629


In [9]:
train_auc = roc_auc_score(
    y_tr, lgbm_enc_no_scale.predict_proba(X_tr_encoded)[:, 1]
)
print(f"Train AUC: {train_auc:.4f}")

Train AUC: 0.9628


In [10]:
train_auc_2 = roc_auc_score(
    y_tr, lgbm_num_no_scale.predict_proba(X_tr_num)[:, 1]
)
print(f"Train AUC: {train_auc:.4f}")

Train AUC: 0.9628


In [10]:
auc = report(lgbm_enc, X_val_encoded, y_val)

Validation ROC-AUC : 0.9573
Validation accuracy: 0.9245

Confusion matrix:
 [[73464  4395]
 [ 6174 55894]] 

              precision    recall  f1-score   support

           0       0.92      0.94      0.93     77859
           1       0.93      0.90      0.91     62068

    accuracy                           0.92    139927
   macro avg       0.92      0.92      0.92    139927
weighted avg       0.92      0.92      0.92    139927



In [11]:
auc = report(lgbm_num, X_val_num, y_val)

Validation ROC-AUC : 0.9574
Validation accuracy: 0.9242

Confusion matrix:
 [[73368  4491]
 [ 6109 55959]] 

              precision    recall  f1-score   support

           0       0.92      0.94      0.93     77859
           1       0.93      0.90      0.91     62068

    accuracy                           0.92    139927
   macro avg       0.92      0.92      0.92    139927
weighted avg       0.92      0.92      0.92    139927



In [11]:
auc = report(lgbm_enc_no_scale, X_val_encoded, y_val)

Validation ROC-AUC : 0.9573
Validation accuracy: 0.9244

Confusion matrix:
 [[73467  4392]
 [ 6184 55884]] 

              precision    recall  f1-score   support

           0       0.92      0.94      0.93     77859
           1       0.93      0.90      0.91     62068

    accuracy                           0.92    139927
   macro avg       0.92      0.92      0.92    139927
weighted avg       0.92      0.92      0.92    139927



In [12]:
auc = report(lgbm_num_no_scale, X_val_num, y_val)

Validation ROC-AUC : 0.9573
Validation accuracy: 0.9243

Confusion matrix:
 [[73364  4495]
 [ 6100 55968]] 

              precision    recall  f1-score   support

           0       0.92      0.94      0.93     77859
           1       0.93      0.90      0.91     62068

    accuracy                           0.92    139927
   macro avg       0.92      0.92      0.92    139927
weighted avg       0.92      0.92      0.92    139927



- Without scaling, not much difference, in fact there is a slight drop - seems negligible to me.

### Saving Model

In [12]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump({"state": state_ohe, "model": lgbm_enc}, MODEL_DIR / "lightgbm_encoded.joblib")
joblib.dump({"state": state_num, "model": lgbm_num}, MODEL_DIR / "lightgbm_numeric.joblib")

['C:\\Users\\Bharat\\Desktop\\remaining-2026\\Career-Tracks\\Track 01 - Data Science\\Relevant Projects\\airline-satisfaction-prediction\\models\\lightgbm_numeric.joblib']

In [13]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump({"state": state_ohe, "model": lgbm_enc_no_scale}, MODEL_DIR / "lightgbm_encoded_no.joblib")
joblib.dump({"state": state_num, "model": lgbm_num_no_scale}, MODEL_DIR / "lightgbm_numeric_no.joblib")

['C:\\Users\\Bharat\\Desktop\\remaining-2026\\Career-Tracks\\Track 01 - Data Science\\Relevant Projects\\airline-satisfaction-prediction\\models\\lightgbm_numeric_no.joblib']